In [1]:
import pandas as pd
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from scipy.special import softmax

import torch
from torch.utils.data import Dataset

import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

SEED = 42
MAX_LEN = 128

In [2]:
MODELS = [
    "allegro/herbert-base-cased",
    "dkleczek/bert-base-polish-cased-v1",
    "sdadas/polish-roberta-base-v2"
]

In [3]:
df = pd.read_csv("hate_train.csv")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

## Ważenie przykładów

In [4]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([0.5463, 5.8972])


In [5]:
from torchvision.ops import sigmoid_focal_loss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )


        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

## Focal Loss
Wyszło że trochę lepszy dla mocno niezbalansowanego zbioru

In [ ]:
import torch.nn.functional as F

class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


class FocalTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(
            alpha=0.75,
            gamma=2.0
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss = self.focal_loss(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [7]:

class HateDataset(Dataset):

    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )

        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    probs = softmax(pred.predictions, axis=1)[:, 1]
    preds = probs >= 0.5

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs)
    }

## Porównanie modeli

In [ ]:
results = []

for model_name in MODELS:

    print("=" * 50)
    print(model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

    val_dataset = HateDataset(
        val_df["sentence"],
        val_df["label"].values,
        tokenizer
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./tmp_{model_name.split('/')[-1]}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=20,
        report_to="none"
    )

    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate()

    results.append({
        "model": model_name,
        "f1": metrics["eval_f1"],
        "accuracy": metrics["eval_accuracy"]
    })

In [ ]:

results_df = pd.DataFrame(results)

print("\nRESULTS")
print(results_df.sort_values("f1", ascending=False))

best_model_name = results_df.sort_values(
    "f1",
    ascending=False
).iloc[0]["model"]

print(f"\nBEST MODEL: {best_model_name}")

## Analiza najlepszego modelu

In [ ]:
## Ostatecznie:
best_model_name = "allegro/herbert-base-cased"

In [34]:
print(best_model_name)

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

val_dataset = HateDataset(
    val_df["sentence"],
    val_df["label"].values,
    tokenizer
)

model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir=f"./best_{best_model_name.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = FocalTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate()



dkleczek/bert-base-polish-cased-v1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dkleczek/bert-base-polish-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	tho

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Specificity,F1,Roc Auc,Pr Auc
1,0.033013,0.035853,0.931807,0.648649,0.423529,0.978793,0.512456,0.921207,0.589821
2,0.016059,0.053383,0.937282,0.724490,0.417647,0.985318,0.529851,0.904801,0.589203
3,0.010332,0.080756,0.935291,0.678571,0.447059,0.980424,0.539007,0.880322,0.579272
4,0.001775,0.103095,0.932802,0.644628,0.458824,0.976618,0.536082,0.869977,0.568143
5,0.000052,0.134305,0.936784,0.721649,0.411765,0.985318,0.524345,0.863794,0.562742
6,0.000962,0.132797,0.936784,0.686957,0.464706,0.980424,0.554386,0.888210,0.589095
7,0.000110,0.137315,0.935291,0.661290,0.482353,0.977162,0.557823,0.884330,0.554012
8,0.000013,0.150932,0.932802,0.619048,0.535294,0.969549,0.574132,0.885449,0.567931
9,0.001073,0.167323,0.934296,0.661017,0.458824,0.978249,0.541667,0.879090,0.572089
10,0.000010,0.172779,0.934793,0.669565,0.452941,0.979337,0.540351,0.881752,0.574709


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,Specificity,F1,Roc Auc,Pr Auc
0.000001,0.150932,15,0.932802,0.619048,0.535294,0.969549,0.574132,0.885449,0.567931


In [40]:
print(metrics)

{'eval_loss': 0.15093237161636353, 'eval_accuracy': 0.9328023892483823, 'eval_precision': 0.6190476190476191, 'eval_recall': 0.5352941176470588, 'eval_specificity': 0.9695486677542142, 'eval_f1': 0.5741324921135647, 'eval_roc_auc': 0.885449253110706, 'eval_pr_auc': 0.5679314261664763}


In [41]:
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96      1839
           1       0.62      0.54      0.57       170

    accuracy                           0.93      2009
   macro avg       0.79      0.75      0.77      2009
weighted avg       0.93      0.93      0.93      2009



Model ma niskie "recall". I tak lepiej niż bez focal loss. Bardziej nie udało się poprawić.

In [37]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

print(cm_df)

          Predicted 0  Predicted 1
Actual 0         1783           56
Actual 1           79           91


## Najlepszy model

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

full_dataset = HateDataset(
    df["sentence"],
    df["label"].values,
    tokenizer
)

best_model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir=f"./finaL_best_{best_model_name.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = FocalTrainer(
    model=best_model,
    args=args,
    train_dataset=full_dataset
)

trainer.train()

## Predykcje

In [39]:
with open("hate_test_data.txt", "r", encoding="utf8") as f:
    test_texts = [line.strip() for line in f]

enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

enc = {
    k: v.to(device)
    for k, v in enc.items()
}

with torch.no_grad():
    outputs = model(**enc)

preds = outputs.logits.argmax(dim=1).cpu().numpy()

pd.DataFrame(preds).to_csv(
    "pred.csv",
    header=False,
    index=False
)
